<a class='anchor' id='toc'></a>
<h1 style='color: #1E90FF; font-size: 2.5em; font-weight: bold;'>FIFA 21 Player Market Value - SparkSQL Analysis</h1>

**Group 10**
- Bárbara Franco (20250388)
- Catarina Mendinhas (20250422)
- Maria Miguel Fonseca (20250380)
- Rodrigo Santos (20250387)
- Rodrigo Teixeira (20250393)

**GitHub repository:**  https://github.com/mariamiguel720/Big-Data-Analytics-Project-25-26

### <font color='#1E90FF'>**Table of Contents**</font>

- [1. Setup & Load Data](#1)
- [2. Distributions](#2)
    - [2.1. Value Distribution](#2_1)
    - [2.2. Wage Distribution](#2_2)
    - [2.3. Age Distribution](#2_3)
    - [2.4. Height Distribution](#2_4)
    - [2.5. Weight Distribution](#2_5)
- [3. Top Clubs & Nationalities](#3)
    - [3.1. Club Analysis](#3_1)
    - [3.2. Top Clubs by Average Value](#3_2)
    - [3.3. Top Nationalities](#3_3)
- [4. Correlations](#4)
- [5. Analysis by Position](#5)
    - [5.1. Most Played Positions](#5_1)
    - [5.2. Average Value by Position](#5_2)
    - [5.3. Physical Profile by Position](#5_3)
- [6. Hits - Most Searched Players](#6)
- [7. Contract Type Analysis](#7)
- [8. Preferred Foot Analysis](#8)
- [9. Total Stats & Base Stats](#9)

<a class='anchor' id='1'></a>
# **1. Setup & Load Data**

[Back to TOC](#toc)
 

In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .master("local[4]") \
    .appName("FIFA21 SparkSQL Analysis") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session ready!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/05 11:30:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/05 11:30:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Session ready!


In [2]:
fifa = spark.read.parquet("../../data/FIFA/cleaned_fifa.parquet")

# Register as a SQL temp view to allow SparkSQL queries
fifa.createOrReplaceTempView("fifa")

print(f"Rows: {fifa.count()}")
print(f"Columns: {len(fifa.columns)}")
fifa.printSchema()

26/06/05 11:31:49 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows: 18979
Columns: 75
root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- OVA: integer (nullable = true)
 |-- POT: integer (nullable = true)
 |-- Club: string (nullable = true)
 |-- Positions: string (nullable = true)
 |-- Height: integer (nullable = true)
 |-- Weight: integer (nullable = true)
 |-- Preferred_Foot: string (nullable = true)
 |-- BOV: integer (nullable = true)
 |-- Best_Position: string (nullable = true)
 |-- Value: integer (nullable = true)
 |-- Wage: integer (nullable = true)
 |-- Release_Clause: integer (nullable = true)
 |-- Attacking: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Finishing: integer (nullable = true)
 |-- Heading_Accuracy: integer (nullable = true)
 |-- Short_Passing: integer (nullable = true)
 |-- Volleys: integer (nullable = true)
 |-- Skill: integer (nullable = true)
 |-- Dribbling: integer (nullable = true)
 |-- 

<a class='anchor' id='2'></a>

# **2. Distributions**

[Back to TOC](#toc) 

In this section we analyse the distribution of the main financial and demographic variables in the dataset, to understand how players are spread across market value, wage, and age ranges.

> **Note on `.toPandas()`**: All visualizations use `.toPandas()` only on small aggregated results returned by SparkSQL. The full DataFrame is never collected to the driver, in a real big data context, the aggregation would still run distributed across the cluster.

<a class='anchor' id='2_1'></a>

## **2.1. Value Distribution**

[Back to TOC](#toc) 
 
We analyse how players are distributed across market value ranges using bucketing in SparkSQL. We also compute descriptive statistics (mean, median, percentiles) to understand the spread of values.

**What does the market value distribution look like? Are most players worth very little, with a few stars worth hundreds of millions?**

In [41]:
# Bucket Value into ranges using SparkSQL for a histogram-like view
value_dist = spark.sql("""
    SELECT
        CASE
            WHEN Value = 0   THEN '= 0'
            WHEN Value < 10000   THEN '< 10K'
            WHEN Value < 100000  THEN '10K - 100K'
            WHEN Value < 500000  THEN '100K - 500K'
            WHEN Value < 1000000  THEN '500K - 1M'
            WHEN Value < 5000000  THEN '1M - 5M'
            WHEN Value < 10000000 THEN '5M - 10M'
            WHEN Value < 25000000  THEN '10M - 25M'
            WHEN Value < 50000000  THEN '25M - 50M'
            WHEN Value < 100000000 THEN '50M - 100M'
            ELSE '> 100M'
        END AS Value_Range,
        COUNT(*) AS Player_Count
    FROM fifa
    WHERE Value IS NOT NULL 
    GROUP BY Value_Range
    ORDER BY MIN(Value)
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    value_dist, x="Value_Range", y="Player_Count",
    title="Distribution of Player Market Values",
    labels={"Value_Range": "Market Value (€)", "Player_Count": "Number of Players"},
    color="Player_Count", color_continuous_scale="Blues"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Yes, the distribution is heavily right skewed. The vast majority of players are valued
between €1M and €5M, while only a small fraction reaches the higher brackets (>€25M).

**What are the key summary statistics for player market value: mean, median, and top percentiles?**

In [42]:
# Summary stats for Value
spark.sql("""
    SELECT
        ROUND(AVG(Value), 2)    AS avg_value,
        ROUND(MIN(Value), 2)    AS min_value,
        ROUND(MAX(Value), 2)    AS max_value,
        PERCENTILE(Value, 0.50) AS median_value,
        PERCENTILE(Value, 0.75) AS p75_value,
        PERCENTILE(Value, 0.90) AS p90_value
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0
""").show()

+----------+---------+---------+------------+---------+---------+
| avg_value|min_value|max_value|median_value|p75_value|p90_value|
+----------+---------+---------+------------+---------+---------+
|2902996.43|     9000|185500000|    975000.0|2000000.0|5500000.0|
+----------+---------+---------+------------+---------+---------+



The average player value is ~€2.9M, but the median is only €975K, less than a third of the mean.
This gap confirms the skew seen above: a small number of elite players pull the average up significantly.

<a class='anchor' id='2_2'></a>

## **2.2. Wage Distribution**

[Back to TOC](#toc) 

Distribution of players' weekly wages grouped into ranges. This helps identify whether the vast majority of players earn relatively little, with a long tail of very high earners.

**How are weekly wages distributed? Is there a long tail of very high earners?**

In [43]:
wage_dist = spark.sql("""
    SELECT
        CASE
            WHEN Wage = 0  THEN '= 0'
            WHEN Wage < 1000  THEN '< 1K'
            WHEN Wage < 10000  THEN '< 10K'
            WHEN Wage < 50000  THEN '10K - 50K'
            WHEN Wage < 100000 THEN '50K - 100K'
            WHEN Wage < 200000 THEN '100K - 200K'
            ELSE '> 200K'
        END AS Wage_Range,
        COUNT(*) AS Player_Count
    FROM fifa
    WHERE Wage IS NOT NULL
    GROUP BY Wage_Range
    ORDER BY MIN(Wage)
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    wage_dist, x="Wage_Range", y="Player_Count",
    title="Distribution of Player Weekly Wages",
    labels={"Wage_Range": "Weekly Wage (€)", "Player_Count": "Number of Players"},
    color="Player_Count", color_continuous_scale="Greens"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Weekly wages follow the same skewed pattern as market values. The majority of players
earn under €10K per week, with very few reaching the higher brackets (>€50K).


<a class='anchor' id='2_3'></a>
 
## **2.3. Age Distribution**

[Back to TOC](#toc) 

Age distribution of players in the dataset. We complement this with average market value by age, to identify at what age players tend to reach their peak market value.

**How old are most FIFA 21 players?**

In [44]:
age_dist = spark.sql("""
    SELECT Age, COUNT(*) AS Player_Count
    FROM fifa
    WHERE Age IS NOT NULL
    GROUP BY Age
    ORDER BY Age
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    age_dist, x="Age", y="Player_Count",
    title="Age Distribution of FIFA 21 Players"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()


The age distribution peaks between 21 and 25, reflecting the typical career stage
at which most professional players are active in the game.

**At what age do players tend to reach their peak market value?**

In [45]:
# Average Value by Age - does value peak at a certain age?
value_by_age = spark.sql("""
    SELECT Age, ROUND(AVG(Value), 0) AS Avg_Value, COUNT(*) AS Player_Count
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0 AND Age IS NOT NULL
    GROUP BY Age
    ORDER BY Age
""").toPandas()  # NOTE: small aggregated result only

fig = px.line(
    value_by_age, x="Age", y="Avg_Value",
    title="Average Market Value by Age",
    labels={"Avg_Value": "Average Value (€)"},
    markers=True
)
fig.show()

Market value rises sharply from age 16, peaks around **27–28**, and declines steadily
after 30. The drop after 35 is particularly steep, as players at that age are typically
approaching the end of their careers. The irregular fluctuations above 40 are due to
the very small number of players at that age in the dataset.

<a class='anchor' id='2_4'></a>

## **2.4. Height Distribution**

[Back to TOC](#toc) 

Distribution of player heights in the dataset, normalised to centimetres during preprocessing.

**How are player heights distributed across the dataset?**

In [73]:
# 2.4 Height Distribution
height_dist = spark.sql("""
    SELECT Height, COUNT(*) AS Player_Count
    FROM fifa
    WHERE Height IS NOT NULL
    GROUP BY Height
    ORDER BY Height
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    height_dist, x="Height", y="Player_Count",
    title="Height Distribution of FIFA 21 Players",
    labels={"Height": "Height (cm)", "Player_Count": "Number of Players"},
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Height follows a roughly normal distribution centred around **180 cm**, which aligns
with the average height of professional footballers. Very few players fall below 160 cm
or above 200 cm.

<a class='anchor' id='2_5'></a>

## **2.5. Weight Distribution**

[Back to TOC](#toc) 

Distribution of player weights in the dataset, normalised to kilograms during preprocessing.

**How are player weights distributed across the dataset?**

In [74]:
# 2.5 Weight Distribution
weight_dist = spark.sql("""
    SELECT Weight, COUNT(*) AS Player_Count
    FROM fifa
    WHERE Weight IS NOT NULL
    GROUP BY Weight
    ORDER BY Weight
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    weight_dist, x="Weight", y="Player_Count",
    title="Weight Distribution of FIFA 21 Players",
    labels={"Weight": "Weight (kg)", "Player_Count": "Number of Players"},
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Weight also follows a roughly normal distribution, centred around **70–75 kg**,
with very few players below 55 kg or above 100 kg.

<a class='anchor' id='3'></a>
 
# **3. Top Clubs & Nationalities**

[Back to TOC](#toc) 

Analysis of the most represented clubs and nationalities in the dataset, crossed with metrics such as average value, OVA, and wage.

<a class='anchor' id='3_1'></a>
 
## **3.1. Club Analysis**

[Back to TOC](#toc) 

We analyse the clubs with the highest average player value and total wage bill. Clubs with fewer than 10 players are filtered out to ensure representativeness.

**Which clubs have the highest average player quality (OVA)**

In [47]:
# --- Top Clubs by Average OVA ---
top_clubs_ova = spark.sql("""
    SELECT
        Club,
        ROUND(AVG(OVA), 2)  AS Avg_OVA,
        COUNT(*)            AS N_Players
    FROM fifa
    WHERE Club IS NOT NULL
    GROUP BY Club
    HAVING N_Players >= 10
    ORDER BY Avg_OVA DESC
    LIMIT 15
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    top_clubs_ova, x="Avg_OVA", y="Club",
    orientation="h",
    title="Top 15 Clubs by Average Overall Rating (min. 10 players)",
    labels={"Avg_OVA": "Average OVA", "Club": ""},
    color_continuous_scale="RdYlGn",
    text="Avg_OVA"
)
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**FC Bayern München** leads in average player quality (OVA: 81.5), followed closely by
Real Madrid and Inter.

**Which clubs have the highest total weekly wage bill?**

In [48]:
# --- Total Wage Bill by Club (Top 10) ---
wage_bill = spark.sql("""
    SELECT
        Club,
        SUM(Wage)            AS Total_Wage_Bill,
        COUNT(*)             AS N_Players
    FROM fifa
    WHERE Club IS NOT NULL AND Wage IS NOT NULL AND Wage > 0
    GROUP BY Club
    HAVING N_Players >= 10
    ORDER BY Total_Wage_Bill DESC
    LIMIT 15
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    wage_bill, x="Total_Wage_Bill", y="Club",
    orientation="h",
    title="Top 15 Clubs by Total Weekly Wage Bill",
    labels={"Total_Wage_Bill": "Total Weekly Wages (€)", "Club": ""},
    text="Total_Wage_Bill",
    hover_data=["N_Players"]
)
fig.update_traces(texttemplate='€%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

On the wage side, **Real Madrid** and **FC Barcelona** have by far the highest weekly
wage bills (€4.7M and €4.1M respectively).

<a class='anchor' id='3_2'></a>
 
## **3.2. Top Clubs by Average Value**

[Back to TOC](#toc) 

We complement the analysis with a ranking of clubs by average and total squad value, coloured by average OVA to cross-reference quality with investment.

**Which clubs have the highest average player market value?**

In [49]:
top_clubs_value = spark.sql("""
    SELECT
        Club,
        ROUND(AVG(Value), 0)  AS Avg_Value,
        ROUND(AVG(OVA), 2)    AS Avg_OVA,
        COUNT(*)              AS N_Players
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0
    GROUP BY Club
    HAVING N_Players >= 10
    ORDER BY Avg_Value DESC
    LIMIT 15
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    top_clubs_value, x="Avg_Value", y="Club",
    orientation="h",
    title="Top 15 Clubs by Average Player Market Value (min. 10 players)",
    labels={"Avg_Value": "Average Value (€)", "Club": ""},
    color="Avg_OVA", color_continuous_scale="RdYlGn",
    text="Avg_Value"
)
fig.update_traces(texttemplate='€%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In terms of average player market value, **Bayern München** tops the list again at
€41M per player, with Real Madrid and Liverpool close behind.


**Which clubs have the most valuable squads in total?**

In [50]:
# Top clubs by total squad value
top_clubs_total = spark.sql("""
    SELECT
        Club,
        ROUND(SUM(Value), 0) AS Total_Squad_Value,
        COUNT(*)             AS N_Players
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0
    GROUP BY Club
    HAVING N_Players >= 10
    ORDER BY Total_Squad_Value DESC
    LIMIT 15
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    top_clubs_total, x="Total_Squad_Value", y="Club",
    orientation="h",
    title="Top 15 Clubs by Total Squad Value",
    labels={"Total_Squad_Value": "Total Squad Value (€)"},
    color="Total_Squad_Value", color_continuous_scale="Blues"
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})

fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

For total squad value, **Liverpool** leads with over €1.2B, narrowly ahead of Manchester
City and Real Madrid. This reflects squad depth rather than individual star quality.

<a class='anchor' id='3_3'></a>
 
## **3.3. Top Nationalities**

[Back to TOC](#toc) 

Ranking of the most represented nationalities in the dataset, alongside each country's average OVA. The world map provides a geographical view of where players come from.

**Which nationalities are most represented in FIFA 21, and how does quality vary by country?**

In [51]:
top_nationalities = spark.sql("""
    SELECT
        Nationality,
        COUNT(*)              AS N_Players,
        ROUND(AVG(OVA), 2)   AS Avg_OVA,
        ROUND(AVG(Value), 0) AS Avg_Value
    FROM fifa
    GROUP BY Nationality
    ORDER BY N_Players DESC
    LIMIT 20
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    top_nationalities, x="Nationality", y="N_Players",
    title="Top 20 Nationalities by Number of Players",
    labels={"N_Players": "Number of Players"},
    color="Avg_OVA", color_continuous_scale="RdYlGn"
)
fig.show()

**England** is by far the most represented nationality (~1,700 players), followed by Germany,
Spain, France, Argentina, and Brazil. However, sheer volume does not equal quality,
**Brazil** and **Argentina** stand out with noticeably higher average OVA (darker green)
despite having fewer players than England or Germany, reflecting the concentration of
elite talent from South America.

**Where in the world do FIFA 21 players come from?**

In [52]:
# World map of player counts by nationality
all_nationalities = spark.sql("""
    SELECT 
        CASE 
            WHEN Nationality IN ('England', 'Scotland', 'Wales', 'Northern Ireland') THEN 'United Kingdom'
            WHEN Nationality = 'Korea Republic' THEN 'South Korea'
            WHEN Nationality = 'Republic of Ireland' THEN 'Ireland'
            WHEN Nationality = 'China PR' THEN 'China'
            ELSE Nationality 
        END AS Map_Country,
        COUNT(*) AS N_Players, 
        ROUND(AVG(OVA), 2) AS Avg_OVA
    FROM fifa
    GROUP BY Map_Country
    ORDER BY N_Players DESC
""").toPandas() # NOTE: small aggregated result only

fig = px.choropleth(
    all_nationalities,
    locations="Map_Country",     
    locationmode="country names",
    color="N_Players",
    title="FIFA 21 Players by Country",
    color_continuous_scale="Blues",
    template="plotly_white"
)

fig.update_traces(marker_line_color='black', marker_line_width=0.5)

fig.show()

/tmp/ipykernel_2884/2824250622.py:18: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


The world map confirms that player representation is heavily concentrated in **Western Europe**
and **South America**, with the rest of the world contributing comparatively few players.

<a class='anchor' id='4'></a>
 
# **4. Correlations**

[Back to TOC](#toc) 


We compute a Pearson correlation matrix across a selected set of key features using 
Spark MLlib's distributed `Correlation` function. This gives a full picture of how variables relate to each other, helping 
identify redundant features and multicollinearity ahead of the ML pipeline in Notebook 3.

**Which features are most correlated with each other, and with player market value?**

In [53]:
corr_vars = [
    "OVA","BOV", "POT", "Total_Stats", "Base_Stats",
    "Dribbling", "Finishing", "Sprint_Speed", "Reactions", "Weak_Foot", "Skill_Moves",
    "Wage", "Release_Clause",
    "Age", "Hits", "N_Positions", "Height", "Weight",
    "Value" 
]

# NOTE: VectorAssembler + Correlation runs fully distributed, big data safe
assembler = VectorAssembler(inputCols=corr_vars, outputCol="features", handleInvalid="skip")
df_vec = assembler.transform(fifa).select("features")

corr_matrix = Correlation.corr(df_vec, "features", method="pearson").head()[0]
corr_array = corr_matrix.toArray()

corr_df = pd.DataFrame(corr_array, index=corr_vars, columns=corr_vars)

fig = px.imshow(
    corr_df,
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Matrix — Key Features",
    text_auto=".2f"
)
fig.update_layout(width=800, height=800)
fig.show()

The correlation matrix reveals several strong relationships. **OVA and BOV** are nearly
identical (0.98), and both correlate strongly with **Reactions** (0.84), suggesting
overall rating is heavily influenced by a player's in game responsiveness.
**Wage and Release_Clause** are almost perfectly correlated with each other (0.82)
and both show strong correlation with **Value** (0.81 and 0.97), which is expected
since all three are financially linked. These columns are excluded from the ML models
to avoid data leakage.

To get a clearer view of which attributes drive market value, we compute the Pearson 
correlation between Value and each feature individually.
Features are grouped by category (Performance, Technical, Financial, and Profile).

**Which attributes drive market value the most, broken down by category?**

In [ ]:
categories = {
    "Performance": ["OVA", "BOV", "POT", "Total_Stats", "Base_Stats"],
    "Technical":   ["Dribbling", "Finishing", "Sprint_Speed", "Reactions", "Weak_Foot", "Skill_Moves"],
    "Financial":   ["Wage", "Release_Clause"],
    "Profile":     ["Age", "Hits", "N_Positions", "Height", "Weight"],
}

all_cols = [(c, cat) for cat, cols in categories.items() for c in cols]

select_exprs = ", ".join([
    f"ROUND(CORR(Value, {c}), 4) AS corr_{c}"
    for c, _ in all_cols
])

query = f"""
    SELECT {select_exprs}
    FROM fifa
    WHERE Value IS NOT NULL
"""

# Spark computes CORR() fully distributed — big data safe
result_row = spark.sql(query).collect()[0]

data_list = []
for feature_name, category_name in all_cols:
    corr_value = result_row[f"corr_{feature_name}"] 
    
    data_list.append({
        "Feature": feature_name,
        "Category": category_name,
        "Correlation_with_Value": corr_value
    })

ext_corr_df = pd.DataFrame(data_list).sort_values("Correlation_with_Value", ascending=True)

fig = px.bar(
    ext_corr_df, 
    x="Correlation_with_Value", 
    y="Feature",
    orientation="h",
    color="Category",
    title="Extended Pearson Correlation with Player Market Value, by Category",
    labels={"Correlation_with_Value": "Correlation", "Feature": ""},
)

fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

The bar chart makes the drivers of market value clearer by category:
- **Financial** features (Release_Clause, Wage) dominate, but are excluded from modelling
- **Performance** attributes (BOV, OVA, POT) are the strongest legitimate predictors
- **Technical** skills (Reactions, Skill_Moves, Dribbling) add moderate signal
- **Profile** attributes (Age, Height, Weight) contribute very little

We visualise the relationship between a player's release clause and their market value. 
Since release clauses are typically set as a multiplier of market value, we expect a near-perfect linear correlation.

In [55]:
release_value = spark.sql("""
    SELECT Release_Clause, Value, Best_Position, Name
    FROM fifa
    WHERE Release_Clause IS NOT NULL AND Release_Clause > 0
    AND Value IS NOT NULL AND Value > 0
""").toPandas()  # NOTE: full scatter — acceptable size (~18k rows)

fig = px.scatter(
    release_value, x="Release_Clause", y="Value",
    color="Best_Position",
    hover_data=["Name"],
    title="Release Clause vs Market Value",
    labels={"Release_Clause": "Release Clause (€)", "Value": "Market Value (€)"},
    opacity=0.6
)
fig.show()

We compare Best Overall Rating (BOV) against market value. Unlike OVA which reflects 
the player's current best position, BOV captures peak performance.

In [56]:
# BOV vs Value scatter — does overall rating explain value?
bov_value = spark.sql("""
    SELECT BOV, Value, Best_Position, Age, Name
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0 AND BOV IS NOT NULL
""").toPandas()  # NOTE: full scatter — acceptable size (~18k rows)

fig = px.scatter(
    bov_value, x="BOV", y="Value",
    color="Best_Position",
    hover_data=["Age", "Name"],
    title="Best Overall vs Market Value",
    labels={"BOV": "Best Overall", "Value": "Market Value (€)"},
    opacity=0.6
)
fig.show()

Reactions was one of the stronger individual technical correlators with value. 
Here we visualise its distribution against market value per position, to understand 
whether this attribute drives value across all positions or only specific roles.

In [57]:
reactions = spark.sql("""
    SELECT Reactions, Value, Best_Position, Name
    FROM fifa
    WHERE Reactions IS NOT NULL AND Reactions > 0
    AND Value IS NOT NULL AND Value > 0
""").toPandas()  # NOTE: full scatter — acceptable size (~18k rows)

fig = px.scatter(
    reactions, x="Reactions", y="Value",
    color="Best_Position",
    hover_data=["Name"],
    title="Reactions vs Market Value",
    labels={"Reactions": "Reactions", "Value": "Market Value (€)"},
    opacity=0.6
)
fig.show()

<a class='anchor' id='5'></a>
 
# **5. Analysis by Position**

[Back to TOC](#toc)

We study how positions are distributed in the dataset, which positions have the most valuable players, and what the typical physical profile of each position looks like.

<a class='anchor' id='5_1'></a>
 
## **5.1. Most Played Positions**

[Back to TOC](#toc)

We identify the most common positions in the dataset and analyse the average versatility per position, i.e. how many different positions each player is able to cover.

**Which positions are most common in the dataset?**

In [58]:
# Most common Best_Position in the dataset
pos_count = spark.sql("""
    SELECT Best_Position, COUNT(*) AS N_Players
    FROM fifa
    WHERE Best_Position IS NOT NULL
    GROUP BY Best_Position
    ORDER BY N_Players DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    pos_count, x="Best_Position", y="N_Players",
    title="Most Common Best Position in FIFA 21",
    labels={"Best_Position": "Position", "N_Players": "Number of Players"},
    color="N_Players", color_continuous_scale="Blues"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

**CB (Centre Back)** is the most common position (~3,500 players), followed by ST and CAM.
This reflects the typical squad structure in football, where defensive roles tend to
be more numerous than attacking ones.

**Which positions produce the most versatile players?**

In [59]:
# How many positions does each player cover on average?
versatility = spark.sql("""
    SELECT
        Best_Position,
        ROUND(AVG(N_Positions), 2) AS Avg_N_Positions,
        COUNT(*) AS N_Players
    FROM fifa
    WHERE Best_Position IS NOT NULL AND N_Positions IS NOT NULL
    GROUP BY Best_Position
    ORDER BY Avg_N_Positions DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    versatility, x="Best_Position", y="Avg_N_Positions",
    title="Average Number of Playable Positions by Best Position",
    labels={"Avg_N_Positions": "Avg Positions Played", "Best_Position": "Best Position"},
    color="Avg_N_Positions", color_continuous_scale="Purples"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Regarding versatility, **wingers and midfielders** (RW, LW, LM, RM, CAM) tend to play
the most positions on average (~2), while **GK** is by far the least versatile,
almost always limited to a single position.

<a class='anchor' id='5_2'></a>

## **5.2. Average Value by Position**

[Back to TOC](#toc) 

We compare average market value and average OVA by position to understand which roles are most valued in the transfer market.

**Which positions command the highest average market value?**

In [60]:
value_by_pos = spark.sql("""
    SELECT
        Best_Position,
        ROUND(AVG(Value), 0)  AS Avg_Value,
        ROUND(AVG(OVA), 2)    AS Avg_OVA,
        ROUND(AVG(Wage), 0)   AS Avg_Wage,
        COUNT(*)              AS N_Players
    FROM fifa
    WHERE Value IS NOT NULL AND Value > 0 AND Best_Position IS NOT NULL
    GROUP BY Best_Position
    ORDER BY Avg_Value DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    value_by_pos, x="Best_Position", y="Avg_Value",
    title="Average Market Value by Best Position",
    labels={"Avg_Value": "Average Value (€)", "Best_Position": "Position"},
    color="Avg_OVA", color_continuous_scale="RdYlGn",
    text="N_Players"
)
fig.update_traces(texttemplate='n=%{text}', textposition='outside')
fig.show()

**CF (Centre Forward)** commands the highest average market value (~€7M), despite having
the smallest sample size (n=76), suggesting the few true CFs in the dataset are
elite players.

<a class='anchor' id='5_3'></a>

## **5.3. Physical Profile by Position**

[Back to TOC](#toc) 

We analyse the average physical profile of each position, base stats (PAC, SHO, PAS, DRI, DEF, PHY) and physical measurements (height and weight), to understand the differences between player profiles.

**How do the base stats (PAC, SHO, PAS, DRI, DEF, PHY) differ across positions?**

In [61]:
# Average key stats per position for radar-style comparison
stats_by_pos = spark.sql("""
    SELECT
        Best_Position,
        ROUND(AVG(PAC), 1) AS Avg_PAC,
        ROUND(AVG(SHO), 1) AS Avg_SHO,
        ROUND(AVG(PAS), 1) AS Avg_PAS,
        ROUND(AVG(DRI), 1) AS Avg_DRI,
        ROUND(AVG(DEF), 1) AS Avg_DEF,
        ROUND(AVG(PHY), 1) AS Avg_PHY
    FROM fifa
    WHERE Best_Position IS NOT NULL
    GROUP BY Best_Position
    ORDER BY Best_Position
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    stats_by_pos.melt(id_vars="Best_Position", var_name="Stat", value_name="Avg"),
    x="Best_Position", y="Avg", color="Stat", barmode="group",
    title="Average Base Stats by Position",
    labels={"Avg": "Average Rating", "Best_Position": "Position"}
)
fig.show()

**Are there significant physical differences (height and weight) between positions?**

In [62]:
# Height and Weight by position
phys_by_pos = spark.sql("""
    SELECT
        Best_Position,
        ROUND(AVG(Height), 1) AS Avg_Height,
        ROUND(AVG(Weight), 1) AS Avg_Weight
    FROM fifa
    WHERE Best_Position IS NOT NULL AND Height IS NOT NULL AND Weight IS NOT NULL
    GROUP BY Best_Position
    ORDER BY Avg_Height DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.scatter(
    phys_by_pos, x="Avg_Height", y="Avg_Weight",
    text="Best_Position",
    title="Average Height vs Weight by Position",
    labels={"Avg_Height": "Average Height (cm)", "Avg_Weight": "Average Weight (kg)"}
)
fig.update_traces(textposition='top center')
fig.show()

There are clear physical differences across the pitch. Goalkeepers (GK) and Center Backs (CB) stand out as the tallest and heaviest players, while Wingers (LW, RW) are the shortest and lightest. Strikers (ST) sit right in the middle.

<a class='anchor' id='6'></a>

# **6. Hits - Most Searched Players**

[Back to TOC](#toc) 

`Hits` represents the number of visits to each player's profile on the SoFIFA platform. We analyse who the most popular players are, whether popularity correlates with value and OVA, and how this metric is distributed across the dataset.

**Which players are searched the most on SoFIFA, and are they also the most valuable?**

In [63]:
# Top 20 most searched players
top_hits = spark.sql("""
    SELECT Name, Club, Nationality, OVA, Value, Hits
    FROM fifa
    WHERE Hits IS NOT NULL
    ORDER BY Hits DESC
    LIMIT 20
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    top_hits, x="Hits", y="Name",
    orientation="h",
    title="Top 20 Most Searched Players (Hits)",
    labels={"Hits": "Profile Views", "Name": ""},
    color="OVA", color_continuous_scale="RdYlGn",
    hover_data=["Club", "Nationality", "Value"],
    template="plotly_white" 
)

fig.update_layout(
    yaxis={
        'categoryorder': 'total ascending',
        'tickfont': {'size': 10},  
        'automargin': True         
    }
)

fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

The analysis of the chart clearly demonstrates that the most searched players are neither the most valuable nor the highest-rated. Axel Tuanzebe and Daniel Maldini lead the searches by a significant margin; however, the colors of their bars (yellow and dark red) indicate that they possess quite low overall ratings (OVA) compared to the rest. On the other hand, world-class players with top ratings, represented in dark green (such as Kylian Mbappé and Bruno Fernandes), show a substantially lower volume of profile views. This proves that popularity on the platform is frequently driven by curiosity.

**Does a player's online popularity (Hits) correlate with their market value or rating?**

In [64]:
# Does popularity (Hits) correlate with Value and OVA?
hits_corr = spark.sql("""
    SELECT
        ROUND(CORR(Hits, Value), 4) AS corr_hits_value,
        ROUND(CORR(Hits, OVA), 4)   AS corr_hits_ova,
        ROUND(CORR(Hits, Wage), 4)  AS corr_hits_wage
    FROM fifa
    WHERE Hits IS NOT NULL AND Value IS NOT NULL AND OVA IS NOT NULL
""").show()
# Spark computes correlation fully distributed - big data safe

+---------------+-------------+--------------+
|corr_hits_value|corr_hits_ova|corr_hits_wage|
+---------------+-------------+--------------+
|         0.3704|       0.2326|        0.2928|
+---------------+-------------+--------------+



Based on the table, a player's online popularity (Hits) shows only a weak to moderate positive correlation with the other metrics, a player's internet popularity is not a strong or linear indicator of their actual technical quality or true financial value

**How is player popularity distributed: do most players have very few profile views?**

In [65]:
# Hits distribution - most players have few hits, a few have millions
hits_dist = spark.sql("""
    SELECT
        CASE
            WHEN Hits < 100    THEN '< 100'
            WHEN Hits < 500    THEN '100 - 500'
            WHEN Hits < 1000   THEN '500 - 1K'
            WHEN Hits < 5000   THEN '1K - 5K'
            WHEN Hits < 10000  THEN '5K - 10K'
            ELSE '> 10K'
        END AS Hits_Range,
        COUNT(*) AS Player_Count
    FROM fifa
    WHERE Hits IS NOT NULL
    GROUP BY Hits_Range
    ORDER BY MIN(Hits)
""").toPandas()  # NOTE: small aggregated result only

fig = px.pie(
    hits_dist, names="Hits_Range", values="Player_Count",
    title="Distribution of Player Profile Hits"
)

fig.show()

Most players have very few profile views. The popularity distribution is highly skewed, with the overwhelming majority of players (94.6%) receiving fewer than 100 profile hits. Only a very tiny fraction of players achieve higher visibility.

<a class='anchor' id='7'></a>

# **7. Contract Type Analysis**

[Back to TOC](#toc) 

Analysis of the distribution of contract types (Permanent, On Loan, Free) and their impact on average market value and wage. We also look at the distribution of contract end years to understand when most contracts are set to expire.

**How many players are on permanent contracts vs on loan?**

In [66]:
# Count by contract status
contract_count = spark.sql("""
    SELECT Contract_Status, COUNT(*) AS N_Players
    FROM fifa
    WHERE Contract_Status IS NOT NULL
    GROUP BY Contract_Status
    ORDER BY N_Players DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.pie(
    contract_count, names="Contract_Status", values="N_Players",
    title="Players by Contract Type"
)

fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Shows that the overwhelming majority of players (93.4%) hold Permanent contracts. Only a small minority are On Loan (5.34%), and an even smaller fraction are Free agents (1.25%).

**Do players on different contract types earn different average market values?**

In [67]:
# Average Value and Wage by contract type
contract_value = spark.sql("""
    SELECT
        Contract_Status,
        ROUND(AVG(Value), 0) AS Avg_Value,
        ROUND(AVG(OVA), 2)   AS Avg_OVA,
        COUNT(*)             AS N_Players
    FROM fifa
    WHERE Contract_Status IS NOT NULL AND Value IS NOT NULL
    GROUP BY Contract_Status
    ORDER BY Avg_Value DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    contract_value.melt(id_vars="Contract_Status", value_vars=["Avg_Value"],
                        var_name="Metric", value_name="Amount"),
    x="Contract_Status", y="Amount", color="Metric", barmode="group",
    title="Average Value by Contract Type",
    labels={"Contract_Status": "Contract Type", "Amount": "€"}
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Players on Permanent and On Loan contracts hold very similar average market values, both sitting just under the €3 million mark, with permanent contracts being only marginally higher. Free agents have a market value of zero.

**Do players on different contract types earn different average wages?**

In [68]:
# Average Value and Wage by contract type
contract_value = spark.sql("""
    SELECT
        Contract_Status,
        ROUND(AVG(Wage), 0)  AS Avg_Wage,
        ROUND(AVG(OVA), 2)   AS Avg_OVA,
        COUNT(*)             AS N_Players
    FROM fifa
    WHERE Contract_Status IS NOT NULL AND Value IS NOT NULL
    GROUP BY Contract_Status
    ORDER BY Avg_Wage DESC
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    contract_value.melt(id_vars="Contract_Status", value_vars=["Avg_Wage"],
                        var_name="Metric", value_name="Amount"),
    x="Contract_Status", y="Amount", color="Metric", barmode="group",
    title="Average Wage by Contract Type",
    labels={"Contract_Status": "Contract Type", "Amount": "€"},
    color_discrete_map={"Avg_Wage": "Orange"  }
)

fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Players "On Loan" earn the highest average wages (exceeding €12,000), followed by players on "Permanent" contracts (approximately €9,000). Players categorized as "Free" agents have an wage of zero.

**When do most permanent contracts expire?**

In [69]:
# Contract end year distribution - when do most contracts expire?
contract_end = spark.sql("""
    SELECT Contract_End_Year, COUNT(*) AS N_Players
    FROM fifa
    WHERE Contract_End_Year IS NOT NULL AND Contract_Status = 'Permanent'
    GROUP BY Contract_End_Year
    ORDER BY Contract_End_Year
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    contract_end, x="Contract_End_Year", y="N_Players",
    title="Permanent Contract Expiry Year Distribution",
    labels={"Contract_End_Year": "Contract End Year", "N_Players": "Number of Players"},
    color="N_Players", color_continuous_scale="Oranges"
)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig.show()

Based on the chart, the vast majority of permanent contracts expire in the year 2021, which records the highest peak with nearly 6,000 players. From that year onwards, there is a clear trend of progressive decrease in the volume of expiring contracts.

<a class='anchor' id='8'></a>

# **8. Preferred Foot Analysis**

[Back to TOC](#toc) 

Analysis of the proportion of right vs left footed players and the impact of preferred foot on technical attributes such as dribbling, finishing, curve, and ball control.

**Are right-footed players more common than left-footed ones?**

In [70]:
# Count by preferred foot
foot_count = spark.sql("""
    SELECT Preferred_Foot, COUNT(*) AS N_Players
    FROM fifa
    WHERE Preferred_Foot IS NOT NULL
    GROUP BY Preferred_Foot
""").toPandas()  # NOTE: small aggregated result only

fig = px.pie(
    foot_count, names="Preferred_Foot", values="N_Players",
    title="Players by Preferred Foot"
)
fig.show()

**Do left-footed players have better technical attributes like dribbling and finishing?**

In [71]:
# Technical attributes by preferred foot - are left-footed players better dribblers?
foot_attrs = spark.sql("""
    SELECT
        Preferred_Foot,
        ROUND(AVG(Dribbling), 2)  AS Avg_Dribbling,
        ROUND(AVG(Finishing), 2)  AS Avg_Finishing,
        ROUND(AVG(Curve), 2)      AS Avg_Curve,
        ROUND(AVG(Ball_Control), 2) AS Avg_Ball_Control,
        ROUND(AVG(Weak_Foot), 2)  AS Avg_Weak_Foot,
        ROUND(AVG(Value), 0)      AS Avg_Value,
        ROUND(AVG(OVA), 2)        AS Avg_OVA
    FROM fifa
    WHERE Preferred_Foot IS NOT NULL
    GROUP BY Preferred_Foot
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    foot_attrs.melt(id_vars="Preferred_Foot",
                    value_vars=["Avg_Dribbling", "Avg_Finishing", "Avg_Curve", "Avg_Ball_Control"],
                    var_name="Attribute", value_name="Avg_Score"),
    x="Attribute", y="Avg_Score", color="Preferred_Foot", barmode="group",
    title="Technical Attributes by Preferred Foot",
    labels={"Avg_Score": "Average Score", "Attribute": ""}
)
fig.show()

We cannot draw any conclusions from this, as there is no significant difference between the skill attributes of left and right footed players.

<a class='anchor' id='9'></a>

# **9. Total Stats**

[Back to TOC](#toc) 

`Total_Stats` is the sum of all individual player attributes. We analyse how this metric relates to market value and how it is distributed across players.

**How many players are in each Total Stats tier - how rare are truly elite players?**

In [72]:
# Total Stats buckets - how many elite players are there?
stats_buckets = spark.sql("""
    SELECT
        CASE
            WHEN Total_Stats < 1500 THEN '< 1500'
            WHEN Total_Stats < 2000 THEN '1500 - 2000'
            WHEN Total_Stats < 2500 THEN '2000 - 2500'
            WHEN Total_Stats < 3000 THEN '2500 - 3000'
            ELSE '> 3000'
        END AS Stats_Range,
        COUNT(*) AS N_Players,
        ROUND(AVG(Value), 0) AS Avg_Value
    FROM fifa
    WHERE Total_Stats IS NOT NULL
    GROUP BY Stats_Range
    ORDER BY MIN(Total_Stats)
""").toPandas()  # NOTE: small aggregated result only

fig = px.bar(
    stats_buckets, x="Stats_Range", y="N_Players",
    title="Player Count by Total Stats Bucket",
    labels={"Stats_Range": "Total Stats Range", "N_Players": "Number of Players"},
    color="Avg_Value", color_continuous_scale="RdYlGn",
    text="N_Players"
)
fig.update_traces(textposition='outside')
fig.show()

Truly elite players are extremely rare. Out of the entire dataset, only 776 players reach the highest tier (2000 - 2500 stats), making them the most valuable on the market. The vast majority of players (12,336) fall into the average middle tier (1500 - 2000 stats), while 5,867 players make up the lowest tier (< 1500 stats).